In [1]:
import torch
import time
import os
from google.colab import drive

# 1. Install required packages
print("Installing/verifying required packages...")
!pip install -q ultralytics pyyaml pandas numpy

# 2. Mount Google Drive
drive.mount('/content/drive')

# 3. Define root and project directory paths pointing to the yolo12s folder
root_dir = "/content/drive/MyDrive/road_crack_detection"
yolo_folder = os.path.join(root_dir, "yolo12s")

os.makedirs(root_dir, exist_ok=True)
os.makedirs(yolo_folder, exist_ok=True)

print(f"Project Root Directory: {root_dir}")
print(f"YOLO12s Output Folder: {yolo_folder}")

# 4. Check CUDA Hardware
print(f"\nPyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

if device.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

Installing/verifying required packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 8.2 MB/s eta 0:00:00
Mounted at /content/drive
Project Root Directory: /content/drive/MyDrive/road_crack_detection
YOLO12s Output Folder: /content/drive/MyDrive/road_crack_detection/yolo12s

PyTorch Version: 2.11.0+cpu
Using Device: cpu


In [2]:
import zipfile
import yaml
import glob

# 1. Define dataset zip path and local extraction path
zip_path = os.path.join(root_dir, "dataa.zip")
extract_to = "/content/dataset"
os.makedirs(extract_to, exist_ok=True)

if os.path.exists(zip_path):
    print(f"Extracting {zip_path} to Colab local runtime...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset extracted successfully.")
else:
    print(f"Error: Could not find 'dataa.zip' at {root_dir}.")

# 2. Locate data.yaml, configure paths, and assign explicit crack class names
data_yaml_path = None
for r, d, files in os.walk(extract_to):
    if "data.yaml" in files:
        data_yaml_path = os.path.join(r, "data.yaml")
        break

if data_yaml_path and os.path.exists(data_yaml_path):
    dataset_root = os.path.dirname(data_yaml_path)
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Re-map absolute paths for runtime training and keep validation split intact
    data_config['path'] = dataset_root
    data_config['train'] = os.path.join(dataset_root, "train/images")
    data_config['val'] = os.path.join(dataset_root, "valid/images")

    # Force explicit class name mappings so plots display text names instead of numbers
    data_config['names'] = {
        0: 'Longitudinal Crack',
        1: 'Transverse Crack',
        2: 'Pothole',
        3: 'Alligator Crack'
    }

    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f)

    print(f"Updated data.yaml configured at: {data_yaml_path}")
    print("Configured Class Names:", data_config.get('names'))
else:
    print("Error: data.yaml could not be found.")

Extracting /content/drive/MyDrive/road_crack_detection/dataa.zip to Colab local runtime...
Dataset extracted successfully.
Updated data.yaml configured at: /content/dataset/data.yaml
Configured Class Names: {0: 'Longitudinal Crack', 1: 'Transverse Crack', 2: 'Pothole', 3: 'Alligator Crack'}


In [3]:
extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp")

def get_all_images_recursive(base_dir):
    all_imgs = []
    if os.path.exists(base_dir):
        for ext in extensions:
            all_imgs.extend(glob.glob(os.path.join(base_dir, "**", ext), recursive=True))
    return all_imgs

train_files = get_all_images_recursive(data_config['train'])
val_files = get_all_images_recursive(data_config['val'])

num_train_raw = len(train_files)
num_val_raw = len(val_files)
total_raw_images = num_train_raw + num_val_raw

def get_folder_size(folder_path):
    total_size = 0
    if os.path.exists(folder_path):
        for dirpath, dirnames, filenames in os.walk(folder_path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.exists(fp):
                    total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

dataset_disk_size_mb = get_folder_size(dataset_root)

print("=== Dataset Size BEFORE Augmentation (Raw Disk) ===")
print(f"Training Images Count:   {num_train_raw}")
print(f"Validation Images Count: {num_val_raw} (Original split preserved)")
print(f"Total Raw Images:        {total_raw_images}")
print(f"Total Disk Storage:      {dataset_disk_size_mb:.2f} MB\n")

batch_size = 8
epochs = 50
total_training_iterations = (num_train_raw // batch_size) * epochs

print("=== Dataset Scale AFTER Augmentation (Dynamic / On-the-Fly) ===")
print(f"Augmentation Type:       Online / Dynamic (Generated per batch)")
print(f"Mosaic Augmentation:     Active (Blends 4 images per sample)")
print(f"Effective Train Samples: {num_train_raw} unique images per epoch")
print(f"Total Training Batches:  ~{total_training_iterations} total iterations across {epochs} epochs")

=== Dataset Size BEFORE Augmentation (Raw Disk) ===
Training Images Count:   5872
Validation Images Count: 1100 (Original split preserved)
Total Raw Images:        6972
Total Disk Storage:      514.81 MB

=== Dataset Scale AFTER Augmentation (Dynamic / On-the-Fly) ===
Augmentation Type:       Online / Dynamic (Generated per batch)
Mosaic Augmentation:     Active (Blends 4 images per sample)
Effective Train Samples: 5872 unique images per epoch
Total Training Batches:  ~36700 total iterations across 50 epochs


In [4]:
augmentation_config = {
    "imgsz": 640,
    "mosaic": 1.0,
    "mixup": 0.1,
    "degrees": 15.0,
    "fliplr": 0.5,
    "flipud": 0.5
}
print("Preprocessing & Augmentation configuration defined.")

Preprocessing & Augmentation configuration defined.


In [5]:
from ultralytics import YOLO

# Load the official pretrained YOLO12 small baseline model
model = YOLO("yolo12s.pt")
print("YOLO12s baseline model initialized successfully.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO12s baseline model initialized successfully.


In [ ]:
import pandas as pd

# Configure training arguments pointing outputs strictly to yolo12s in Google Drive with batch=8 and epochs=50
training_args = {
    "data": data_yaml_path,
    "epochs": 50,
    "imgsz": augmentation_config["imgsz"],
    "batch": 8,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "cache": False,
    "save": True,
    "save_period": 10,
    "patience": 10,
    "project": yolo_folder,    # Strictly targets /road_crack_detection/yolo12s/
    "name": "train_run"
}

print(f"Starting training execution... All outputs will save to: {yolo_folder}/train_run")

start_time = time.time()
results = model.train(**training_args)
training_wall_time = time.time() - start_time

print(f"Training completed in {training_wall_time:.2f} seconds.")

# Export clean training history CSV
run_dir = model.trainer.save_dir
results_csv_path = os.path.join(run_dir, "results.csv")

if os.path.exists(results_csv_path):
    df_results = pd.read_csv(results_csv_path)
    df_results.columns = df_results.columns.str.strip()

    summary_save_path = os.path.join(yolo_folder, "training_validation_metrics_summary.csv")
    df_results.to_csv(summary_save_path, index=False)
    print(f"Metrics exported securely to: {summary_save_path}")

Starting training execution... All outputs will save to: /content/drive/MyDrive/road_crack_detection/yolo12s/train_run
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol

In [ ]:
import shutil
from pathlib import Path
import numpy as np

# Load the best saved checkpoint from the yolo12s folder
best_model_path = os.path.join(model.trainer.save_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)
print(f"Loaded best model checkpoint from: {best_model_path}")

# Run evaluation on the original validation split
val_start_time = time.time()
val_metrics = best_model.val(data=data_yaml_path, split="val")
val_wall_time = time.time() - val_start_time

# Extract performance scores
mean_precision = np.mean(val_metrics.box.p)
mean_recall = np.mean(val_metrics.box.r)
map50 = val_metrics.box.map50
map50_95 = val_metrics.box.map

print("\n--- Final Validation Evaluation Scores ---")
print(f"Mean Precision: {mean_precision:.4f}")
print(f"Mean Recall:    {mean_recall:.4f}")
print(f"mAP@50:         {map50:.4f}")
print(f"mAP@50-95:      {map50_95:.4f}")
print(f"Evaluation Time: {val_wall_time:.2f} seconds")

# Save summary scores CSV inside yolo12s folder
val_scores_df = pd.DataFrame({
    "Metric": ["Mean_Precision", "Mean_Recall", "mAP@50", "mAP@50-95", "Inference_Wall_Time_Sec"],
    "Score": [mean_precision, mean_recall, map50, map50_95, val_wall_time]
})
val_scores_save_path = os.path.join(yolo_folder, "validation_evaluation_scores.csv")
val_scores_df.to_csv(val_scores_save_path, index=False)
print(f"Validation evaluation summary saved to: {val_scores_save_path}")

# Safely locate validation output plots directory and copy to Drive
local_runs_dir = getattr(val_metrics, 'save_dir', None)
if not local_runs_dir or not os.path.exists(str(local_runs_dir)):
    val_runs = sorted(Path("/content/runs/detect").glob("val*"), key=os.path.getmtime)
    local_runs_dir = val_runs[-1] if val_runs else None

drive_plots_dir = os.path.join(yolo_folder, "validation_evaluation_plots")
if local_runs_dir and os.path.exists(local_runs_dir):
    shutil.copytree(local_runs_dir, drive_plots_dir, dirs_exist_ok=True)
    print(f"Evaluation plots and visual logs copied to: {drive_plots_dir}")